# Module 3: Spark on GPU vs. cuDF (The Tooling Debate)

Welcome to the final module of this section! We will compare two key data engineering tools: Spark and cuDF.

**Section Goals:**
* Understand the difference between distributed cluster and single-node processing.
* Query your GPU VRAM limits to find the Out-Of-Memory (OOM) threshold.
* Learn when to choose Spark on GPU vs. cuDF.

### Distributed Cluster vs. Single System

* **RAPIDS cuDF:** Excellent for single-node vertical scaling. Extremely fast because it operates directly in the GPU VRAM of one server. However, it is limited by that GPU's VRAM size (e.g. 16GB). If your dataset is larger than the VRAM, it crashes with an OOM error.
* **Apache Spark (with RAPIDS Accelerator):** Designed for multi-node horizontal scale-out. It distributes data across a cluster. There is significant network communication overhead, but it can scale to Petabytes without running out of memory.

### Visualizing Scaling Styles

Here is the structural difference between horizontal Spark clusters and single-node cuDF acceleration:

![Spark vs cuDF](images/spark-vs-cudf.svg)

### Step 1: Querying GPU VRAM Capacity

Let's write a small Python block using PyTorch to query the total and currently allocated memory on our GPU. This helps us understand our single-node cuDF limits.

In [ ]:
import torch

total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
allocated_mem = torch.cuda.memory_allocated(0) / (1024**3)
print(f"Total GPU memory: {total_mem:.2f} GB")
print(f"Allocated GPU memory: {allocated_mem:.2f} GB")

### Understanding OOM Thresholds

If your dataset size (plus intermediate query states) exceeds the free VRAM space, cuDF will crash. 

**Architectural Decision Guide:**
* Choose **cuDF** if your dataset is under 50GB and fits on a single VM. It is faster and cheaper.
* Choose **Spark on GPU** if your dataset is in Terabytes/Petabytes and requires cluster division, or if you need robust fault tolerance across servers.

### Module 3 Recap

* **cuDF** offers raw, single-node scale-up performance up to VRAM limits.
* **Apache Spark** offers horizontal scale-out for massive cluster jobs.
* Querying VRAM allocation helps data engineers design pipelines that avoid OOM errors.